## Find atomic contacts between antibody and antigen pairs

In this notebook we will create a DataFrame that contains atomic contacts for all the structures in our cleaned summary file.

The desired result is a DataFrame with columns

- `pdb_id`, e.g. '9ds1'
- `ab_chain`, e.g. 'H' 
- `ab_chaintype`, 'heavy' or 'light'
- `ag_resnum`
- `ab_resnumi`, including icode, e.g. '52A'
- `ab_resname`
- `ab_atom`
- `ag_chain`, e.g. 'G'
- `ag_resnum`
- `ag_resnumi`, e.g. '13'
- `ag_resname`, e.g. 'TYR'
- `ag_atom`
- `distance`between the two closest atoms of the ab_residue with the antigen

### Functions

We will first create a function `atomic_contact_points` to find all contact points of antibodies with their antigenes. The function loops over the atoms in ab_chain to get all atoms of ag_chain that are within distance, and reports a DataFrame with columns

The function `residue_occurrence(chain)` provides the occurrence of residue numbers in the heavy and light chains.

In the following, two DataFrames are created with the information on the contacts and the residue occorence. 

Import required libraries

In [5]:
import os.path
from Bio.PDB import PDBParser, NeighborSearch
import pandas as pd
import numpy as np

In [7]:
SUMMARY_FILE = '../../generated/data cleanup/ab_ag_filtered_pdb.tsv'

PDB_DIR = '../../data/pdbs_all_cnf' #all pdb files

### Function - find atomic contact points

The function `atomic_contact_points(ab_chain, ag_chain, distance)` loops over the atoms in ab_chain to get all atoms of ag_chain that are within distance, and reports a DataFrame with columns
- ab_resnum
- ab_icode
- ab_resname
- ab_atom
- ag_resnum
- ag_icode
- ag_resname
- ag_atom
- distance

But only report for residues that are amino acids, i.e. het_flag == ' '.

In [8]:
def atomic_contact_points(ab_chain, ag_chain, distance):
    res = []
    ns = NeighborSearch(list(ag_chain.get_atoms()))
    for ab_atom in ab_chain.get_atoms():
        ab_res = ab_atom.get_parent()
        close_ag_atoms = ns.search(ab_atom.coord, distance)
        for ag_atom in close_ag_atoms:
            ag_res = ag_atom.get_parent()
            if ab_res.id[0] == ' ' and ag_res.id[0] == ' ':
                dist = np.linalg.norm(ab_atom.coord - ag_atom.coord)
                tmp = dict(ab_resnum = ab_res.id[1],
                           ab_icode = ab_res.id[2],
                           ab_resname = ab_res.get_resname(),
                           ab_atom = ab_atom.id,
                           ag_resnum = ag_res.id[1],
                           ag_icode = ag_res.id[2],
                           ag_resname = ag_res.get_resname(),
                           ag_atom = ag_atom.id,
                           distance = dist) 
                res.append(tmp)

    return pd.DataFrame(res)

### Function - find residue occurrence

We are also interested in the occurrence of residue numbers in the heavy and light chains.
The function `residue_occurrence(chain)` creates as output a DataFrame with columns 
- ab_resnum
- ab_icode
- ab_resname

Output is restricted to het_name == ' ' and resnum <= 128



In [9]:
def residue_occurrence(chain):
    results = []
    for res in chain.get_residues():
        if res.id[0] == ' ' and res.id[1] <= 128:
            tmp = dict(ab_resnum = res.id[1],
                       ab_icode = res.id[2],
                       ab_resname = res.get_resname())
            results.append(tmp)

    return pd.DataFrame(results)

In [10]:
summary = pd.read_csv(SUMMARY_FILE, sep='\t')
summary.head()


,pdb,Hchain,Lchain,model,antigen_chain,antigen_type,antigen_name,compound,organism,heavy_species,light_species,antigen_species,resolution,method,scfv,engineered,heavy_subclass,light_subclass,light_ctype,species
0,8veb,G,I,0,E,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.97,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
1,8ved,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E11 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.98,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV2,Kappa,Influenza A
2,8vee,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,3.18,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
3,8vef,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 UCA (unm...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,3.04,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
4,9dpc,H,L,0,D,protein,neuraminidase,Structure of Fab 297 in complex with influenza...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.65,ELECTRON MICROSCOPY,False,True,IGHV1,IGKV1,Kappa,Influenza A


Now its time to loop through our ab_ag_complex structures and calculate the ab_ag_contacts and create DataFrames with the information on the contacts and the residue occorence. 

In [11]:
import sys

contacts = pd.DataFrame()
residues = pd.DataFrame()

for i, row in summary.iterrows():
    pdb_id = row['pdb']
    if pdb_id == '7mtb':
        continue
    hchain = row['Hchain']
    lchain = row['Lchain']
    antigen_chain = row['antigen_chain']

    try:

    
        filename = os.path.join(PDB_DIR, f'{pdb_id}.pdb')
        parser = PDBParser(PERMISSIVE=1)
        structure = parser.get_structure(pdb_id, filename)

        acph = atomic_contact_points(structure[0][hchain], structure[0][antigen_chain], 4.0)
        
        acph.insert(loc = 0, column = 'ag_chain', value = antigen_chain)
        acph.insert(loc = 0, column = 'ab_chain', value = hchain)
        acph.insert(loc = 0, column = 'chain_type', value = 'heavy')
        acph.insert(loc = 0, column = 'pdb_id', value = pdb_id)

        resh = residue_occurrence(structure[0][hchain])
        resh.insert(loc = 0, column = 'ab_chain', value = hchain)
        resh.insert(loc = 0, column = 'chain_type', value = 'heavy')
        resh.insert(loc = 0, column = 'pdb_id', value = pdb_id)


        acpl = atomic_contact_points(structure[0][lchain], structure[0][antigen_chain], 4.0)
        
        acpl.insert(loc = 0, column = 'ag_chain', value = antigen_chain)
        acpl.insert(loc = 0, column = 'ab_chain', value = lchain)
        acpl.insert(loc = 0, column = 'chain_type', value = 'light')
        acpl.insert(loc = 0, column = 'pdb_id', value = pdb_id)

        resl = residue_occurrence(structure[0][lchain])
        resl.insert(loc = 0, column = 'ab_chain', value = lchain)
        resl.insert(loc = 0, column = 'chain_type', value = 'light')
        resl.insert(loc = 0, column = 'pdb_id', value = pdb_id)

        residues = pd.concat([residues, resh, resl])
        contacts = pd.concat([contacts, acph, acpl])

    except Exception as e:
        print(e)
        print(row)
        sys.exit(1)




In [12]:
contacts.to_csv("../../generated/contacts/atomic_contacts.tsv", sep='\t', index = False)
contacts.head()

,pdb_id,chain_type,ab_chain,ag_chain,ab_resnum,ab_icode,ab_resname,ab_atom,ag_resnum,ag_icode,ag_resname,ag_atom,distance
0,8veb,heavy,G,E,31.0,A,GLY,CA,1018.0,,MET,O,3.666781
1,8veb,heavy,G,E,31.0,A,GLY,CA,1019.0,,ASP,OD1,3.963187
2,8veb,heavy,G,E,31.0,A,GLY,C,1018.0,,MET,O,3.722525
3,8veb,heavy,G,E,31.0,B,GLY,N,1018.0,,MET,O,3.997262
4,8veb,heavy,G,E,31.0,B,GLY,N,1019.0,,ASP,OD1,3.729019


In [9]:
residues.to_csv("../../generated/contacts/residues.tsv", sep='\t', index = False)
residues.head()

,pdb_id,chain_type,ab_chain,ab_resnum,ab_icode,ab_resname
0,8veb,heavy,G,1,,GLN
1,8veb,heavy,G,2,,VAL
2,8veb,heavy,G,3,,GLN
3,8veb,heavy,G,4,,LEU
4,8veb,heavy,G,5,,LEU


In [14]:
summary.pdb.nunique()


962

## create dataframe of residue contacts

Convert atomic contact points to residue contact points

In [15]:
rcp = (contacts
 .get(["pdb_id", "chain_type", "ab_chain", "ab_resnum", "ab_icode", "ab_resname"])
 .drop_duplicates()
 .assign(contact = 1))
 

rcp.head(3)

,pdb_id,chain_type,ab_chain,ab_resnum,ab_icode,ab_resname,contact
0,8veb,heavy,G,31.0,A,GLY,1
3,8veb,heavy,G,31.0,B,GLY,1
5,8veb,heavy,G,33.0,,TYR,1


In [16]:
rcp = (contacts
 .groupby(["pdb_id", "chain_type", "ab_chain", "ab_resnum", "ab_icode", "ab_resname"])
 .agg(distance = ("distance", "min"))
 .assign(contact = 1)
 .reset_index())

rcp.head(3)



,pdb_id,chain_type,ab_chain,ab_resnum,ab_icode,ab_resname,distance,contact
0,1adq,heavy,H,1.0,,GLU,2.596962,1
1,1adq,heavy,H,31.0,,ASP,2.896311,1
2,1adq,heavy,H,52.0,A,TRP,3.202886,1


Annotate residue occurrences with contact information

In [17]:
rc = residues.merge(rcp, how = "left", 
          on = ["pdb_id", "chain_type", "ab_chain", "ab_resnum", "ab_icode", "ab_resname"]
          ).fillna(0)

rc.head(3)

,pdb_id,chain_type,ab_chain,ab_resnum,ab_icode,ab_resname,distance,contact
0,8veb,heavy,G,1,,GLN,0.0,0.0
1,8veb,heavy,G,2,,VAL,0.0,0.0
2,8veb,heavy,G,3,,GLN,0.0,0.0


In [18]:
rc.to_csv("../../generated/contacts/residue_contacts_all.tsv", sep="\t", index = False)


residues = pd.read_csv("../../generated/contacts/residues.tsv", sep="\t")
residues.head(3)

contacts = pd.read_csv("../../generated/contacts/atomic_contacts.tsv", sep="\t")
contacts.head(3)


,pdb_id,chain_type,ab_chain,ag_chain,ab_resnum,ab_icode,ab_resname,ab_atom,ag_resnum,ag_icode,ag_resname,ag_atom,distance
0,8veb,heavy,G,E,31.0,A,GLY,CA,1018.0,,MET,O,3.666781
1,8veb,heavy,G,E,31.0,A,GLY,CA,1019.0,,ASP,OD1,3.963188
2,8veb,heavy,G,E,31.0,A,GLY,C,1018.0,,MET,O,3.722525


# Interaktionstypen
## Cut-off references
**VdW-Radii** nach Bondi (1964):
Rowland, R. S., & Taylor, R. (1996). Intermolecular Nonbonded Contact Distances in Organic Crystal Structures: Comparison with Distances Expected from van der Waals Radii. The Journal of Physical Chemistry, 100(18), 7384–7391. https://doi.org/10.1021/jp953141+

**Hydrogen-bonds:**
Kajander T, Kahn PC, Passila SH, Cohen DC, Lehtio L, Adolfsen W, Warwicker J, Schell U, Goldman A. Buried charged surface in proteins. Structure. 2000 Nov 15;8(11):1203-14.

**salt bridges:** https://proteintools.uni-bayreuth.de/salt/documentation#:~:text=The%20distance%20between%20residues%20participating,render%20salt%20bridges%20in%20proteins. 05.07.

**Hydrophobic contacts**:
Onofrio, A., Parisi, G., Punzi, G., Todisco, S., Di Noia, M. A., Bossis, F., Turi, A., De Grassi, A., & Pierri, C. L. (2014). Distance-dependent hydrophobic–hydrophobic contacts in protein folding simulations. Physical Chemistry Chemical Physics, 16(35), 18907–18917. https://doi.org/10.1039/c4cp01131g

**Aromtic contacts**: Wilson, K. A., Kellie, J. L., & Wetmore, S. D. (2014). DNA-protein π-interactions in nature: abundance, structure, composition and strength of contacts between aromatic amino acids and DNA nucleobases or deoxyribose sugar. Nucleic acids research, 42(10), 6726–6741. https://doi.org/10.1093/nar/gku269


In [20]:
import pandas as pd

# Van-der-Waals-Radien nach Bondi (1964)
vdw_radii = {
    "H": 1.20,
    "C": 1.70,
    "N": 1.55,
    "O": 1.52,
    "S": 1.80,
    "F": 1.47,
    "CL": 1.75,
    "BR": 1.85,
    "I": 1.98
}

# Interaktionsklassifikation
def classify_contact(row):
    ab_res, ab_atom = str(row['ab_resname']).upper(), str(row['ab_atom']).upper()
    ag_res, ag_atom = str(row['ag_resname']).upper(), str(row['ag_atom']).upper()
    ab_el = str(row['ab_element']).upper()
    ag_el = str(row['ag_element']).upper()
    dist = row['distance']

    # Wasserstoffbrücke
    hbond_donors = ["OH", "NE", "NE2", "ND1", "NZ", "NH1", "NH2", "OG", "OG1", "ND2"]
    hbond_acceptors = ["OD1", "OD2", "OE1", "OE2", "O", "OG", "OG1", "OH"]
    if dist < 3.5:
        if (ab_atom in hbond_donors and ag_atom in hbond_acceptors) or \
           (ag_atom in hbond_donors and ab_atom in hbond_acceptors):
            return "hydrogen bond"

    # Salzbrücke
    positive = ["ARG", "LYS", "HIS"]
    negative = ["ASP", "GLU"]
    if dist < 4.0:
        if (ab_res in positive and ag_res in negative) or (ab_res in negative and ag_res in positive):
            return "salt bridge"

    # Hydrophobe Kontakte
    hydrophobic_atoms = ["C", "CA", "CB", "CG", "CG1", "CG2", "CD", "CD1", "CD2", "CE", "CE1", "CE2", "CZ", "SD"]
    if ab_atom in hydrophobic_atoms and ag_atom in hydrophobic_atoms and 3.8 < dist < 5.0: #between 3.8 and 5.0 Ångström
        return "hydrophobic"

    # Aromatische Kontakte
    aromatic_residues = ["PHE", "TYR", "TRP", "HIS"]
    if ab_res in aromatic_residues and ag_res in aromatic_residues and dist < 6.0:
        return "pi-pi stacking"


    # Van-der-Waals (präzise)
    r1 = vdw_radii.get(ab_el)
    r2 = vdw_radii.get(ag_el)
    if r1 and r2:
        optimal = r1 + r2
        tolerance = 0.5  # Ångström
        if (optimal - tolerance) <= dist <= (optimal + tolerance):
            return "van der Waals"

    return "none"


# CSV-Datei einlesen (bitte Pfad anpassen)
df_contacts = pd.read_csv("../../generated/contacts/atomic_contacts.tsv", sep="\t")

# Hilfsfunktion zum Extrahieren des Elementsymbols aus dem Atomnamen
def extract_element(atom_name):
    atom_name = atom_name.strip().upper()
    # Check for two-letter elements first
    if atom_name[:2] in vdw_radii:
        return atom_name[:2]
    # Otherwise, use the first letter
    return atom_name[0]

# Elementspalten hinzufügen
df_contacts['ab_element'] = df_contacts['ab_atom'].apply(extract_element)
df_contacts['ag_element'] = df_contacts['ag_atom'].apply(extract_element)

# Neue Spalte für Interaktionstypen
df_contacts['interaction'] = df_contacts.apply(classify_contact, axis=1)

# Ausgabe anzeigen / speichern
print(df_contacts[['ab_resname', 'ab_atom', 'ag_resname', 'ag_atom', 'distance', 'interaction']])

df_contacts


       ab_resname ab_atom ag_resname ag_atom  distance     interaction
0             GLY      CA        MET       O  3.666781   van der Waals
1             GLY      CA        ASP     OD1  3.963188            none
2             GLY       C        MET       O  3.722525            none
3             GLY       N        MET       O  3.997263            none
4             GLY       N        ASP     OD1  3.729019            none
...           ...     ...        ...     ...       ...             ...
144713        ARG       O        LYS      CE  3.926962            none
144714        ARG       O        LYS      NZ  2.934360   hydrogen bond
144715        TRP      CG        PHE     CE1  3.978944     hydrophobic
144716        TRP     CD1        PHE     CE1  3.794080  pi-pi stacking
144717        TRP     CD1        PHE      CZ  3.721476  pi-pi stacking

[144718 rows x 6 columns]


,pdb_id,chain_type,ab_chain,ag_chain,ab_resnum,ab_icode,ab_resname,ab_atom,ag_resnum,ag_icode,ag_resname,ag_atom,distance,ab_element,ag_element,interaction
0,8veb,heavy,G,E,31.0,A,GLY,CA,1018.0,,MET,O,3.666781,C,O,van der Waals
1,8veb,heavy,G,E,31.0,A,GLY,CA,1019.0,,ASP,OD1,3.963188,C,O,none
2,8veb,heavy,G,E,31.0,A,GLY,C,1018.0,,MET,O,3.722525,C,O,none
3,8veb,heavy,G,E,31.0,B,GLY,N,1018.0,,MET,O,3.997263,N,O,none
4,8veb,heavy,G,E,31.0,B,GLY,N,1019.0,,ASP,OD1,3.729019,N,O,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144713,7tas,light,L,E,54.0,,ARG,O,417.0,,LYS,CE,3.926962,O,C,none
144714,7tas,light,L,E,54.0,,ARG,O,417.0,,LYS,NZ,2.934360,O,N,hydrogen bond
144715,7tas,light,L,E,91.0,,TRP,CG,486.0,,PHE,CE1,3.978944,C,C,hydrophobic
144716,7tas,light,L,E,91.0,,TRP,CD1,486.0,,PHE,CE1,3.794080,C,C,pi-pi stacking


In [21]:
# liste der Interaktionstypen
interaction_types = df_contacts['interaction'].unique()
interaction_types.tolist() 

# anteile der Interaktionstypen
interaction_counts = df_contacts['interaction'].value_counts(normalize=True) * 100
print(interaction_counts)

interaction
van der Waals     37.951741
none              34.804240
hydrophobic        8.837878
pi-pi stacking     6.781465
hydrogen bond      6.567255
salt bridge        5.057422
Name: proportion, dtype: float64
